In [ ]:
# ==================================================
# 导入必要的库 - LSTM（长短期记忆网络）实现
# ==================================================
import torch  # PyTorch深度学习框架
import RNN  # 自定义RNN模块，包含数据加载和训练函数
from torch import nn  # PyTorch神经网络模块
from d2l import torch as d2l  # d2l工具库

# LSTM（Long Short-Term Memory，长短期记忆网络）：
# - 专门设计用于解决RNN的长期依赖问题和梯度消失/爆炸问题
# - 使用三个门（输入门、遗忘门、输出门）和一个记忆细胞
# - 能够学习何时记住、何时遗忘、何时输出信息
# - 是处理序列数据最成功的架构之一

batch_size, num_steps = 32, 35  # 批量大小=32，时间步数=35
train_iter, vocab = RNN.load_data_time_machine(batch_size, num_steps)

In [ ]:
# ==================================================
# 初始化LSTM的所有参数
# ==================================================
def get_lstm_params(vocab_size, num_hiddens, device):
    """
    初始化LSTM的所有可训练参数
    
    LSTM需要四组参数：
    1. 输入门(Input Gate) I_t：控制接受多少新信息
    2. 遗忘门(Forget Gate) F_t：控制遗忘多少旧记忆
    3. 输出门(Output Gate) O_t：控制输出多少信息
    4. 候选记忆细胞 C̃_t：待添加到记忆细胞的新信息
    
    参数:
        vocab_size: 词汇表大小
        num_hiddens: 隐藏层单元数
        device: 计算设备
    
    返回:
        包含所有参数的列表
    """
    num_inputs = num_outputs = vocab_size

    def normal(shape):
        # 使用小标准差的正态分布初始化权重
        return torch.randn(size=shape, device=device)*0.01

    def three():
        """
        为每个门创建一组参数：输入权重、隐藏权重、偏置
        """
        return (normal((num_inputs, num_hiddens)),   # 输入到隐藏的权重
                normal((num_hiddens, num_hiddens)),  # 隐藏到隐藏的权重
                torch.zeros(num_hiddens, device=device))  # 偏置

    # ===== LSTM的四组核心参数 =====
    W_xi, W_hi, b_i = three()  # 输入门参数：I_t = σ(X_t·W_xi + H_{t-1}·W_hi + b_i)
    W_xf, W_hf, b_f = three()  # 遗忘门参数：F_t = σ(X_t·W_xf + H_{t-1}·W_hf + b_f)
    W_xo, W_ho, b_o = three()  # 输出门参数：O_t = σ(X_t·W_xo + H_{t-1}·W_ho + b_o)
    W_xc, W_hc, b_c = three()  # 候选记忆元参数：C̃_t = tanh(X_t·W_xc + H_{t-1}·W_hc + b_c)
    
    # ===== 输出层参数 =====
    W_hq = normal((num_hiddens, num_outputs))
    b_q = torch.zeros(num_outputs, device=device)
    
    # 附加梯度，使所有参数可训练
    params = [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o, W_xc, W_hc,
              b_c, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

In [ ]:
# ==================================================
# 初始化LSTM的隐藏状态
# ==================================================
def init_lstm_state(batch_size, num_hiddens, device):
    """
    初始化LSTM的隐藏状态
    
    LSTM需要两个状态：
    1. H (Hidden state): 隐藏状态，传递给下一时间步和输出层
    2. C (Cell state): 记忆细胞状态，存储长期信息
    
    返回:
        包含两个全零张量的元组 (H, C)，形状都是(batch_size, num_hiddens)
    """
    return (torch.zeros((batch_size, num_hiddens), device=device),  # H
            torch.zeros((batch_size, num_hiddens), device=device))  # C

In [ ]:
# ==================================================
# LSTM前向传播核心函数
# ==================================================
def lstm(inputs, state, params):
    """
    LSTM的前向传播计算
    
    LSTM的核心公式（按计算顺序）：
    1. 输入门：I_t = σ(X_t·W_xi + H_{t-1}·W_hi + b_i)
       - 决定接受多少新信息
    
    2. 遗忘门：F_t = σ(X_t·W_xf + H_{t-1}·W_hf + b_f)
       - 决定遗忘多少旧记忆
    
    3. 输出门：O_t = σ(X_t·W_xo + H_{t-1}·W_ho + b_o)
       - 决定输出多少信息
    
    4. 候选记忆细胞：C̃_t = tanh(X_t·W_xc + H_{t-1}·W_hc + b_c)
       - 当前时间步的候选信息
    
    5. 记忆细胞更新：C_t = F_t⊙C_{t-1} + I_t⊙C̃_t
       - 遗忘旧记忆的一部分 + 添加新信息的一部分
    
    6. 隐藏状态更新：H_t = O_t⊙tanh(C_t)
       - 基于更新后的记忆细胞生成隐藏状态
    
    其中：
    - σ是sigmoid函数，输出范围[0,1]，用于门控
    - tanh是双曲正切函数，输出范围[-1,1]，用于候选信息
    - ⊙表示逐元素乘法（Hadamard积）
    
    参数:
        inputs: 输入序列，形状为(时间步数, 批量大小, 词表大小)
        state: (H, C) 元组，包含隐藏状态和记忆细胞状态
        params: 所有LSTM参数
    
    返回:
        outputs: 所有时间步的输出拼接，形状为(时间步数*批量大小, 词表大小)
        (H, C): 最后一个时间步的状态
    """
    [W_xi, W_hi, b_i, W_xf, W_hf, b_f, W_xo, W_ho, b_o, W_xc, W_hc, b_c,
     W_hq, b_q] = params
    (H, C) = state  # 解包隐藏状态和记忆细胞状态
    outputs = []
    
    for X in inputs:  # 遍历每个时间步
        # 计算三个门和候选记忆细胞
        I = torch.sigmoid((X @ W_xi) + (H @ W_hi) + b_i)  # 输入门
        F = torch.sigmoid((X @ W_xf) + (H @ W_hf) + b_f)  # 遗忘门
        O = torch.sigmoid((X @ W_xo) + (H @ W_ho) + b_o)  # 输出门
        C_tilda = torch.tanh((X @ W_xc) + (H @ W_hc) + b_c)  # 候选记忆细胞
        
        # 更新记忆细胞：遗忘旧记忆 + 添加新记忆
        C = F * C + I * C_tilda
        
        # 更新隐藏状态：输出门控制输出多少记忆细胞的信息
        H = O * torch.tanh(C)
        
        # 计算当前时间步的输出
        Y = (H @ W_hq) + b_q
        outputs.append(Y)
    
    # 拼接所有时间步的输出
    return torch.cat(outputs, dim=0), (H, C)

In [ ]:
# ==================================================
# 从零实现的LSTM模型训练
# ==================================================
vocab_size, num_hiddens, device = len(vocab), 256, d2l.try_gpu()
num_epochs, lr = 500, 50  # 训练500轮，学习率50（LSTM通常需要较大的学习率）

# 创建从零实现的LSTM模型
model = RNN.RNNModelScratch(len(vocab), num_hiddens, device, get_lstm_params,
                            init_lstm_state, lstm)

# 开始训练从零实现的LSTM
# LSTM通常能更好地捕获长期依赖关系，困惑度应该比标准RNN更低
RNN.train_ch8(model, train_iter, vocab, lr, num_epochs, device)

In [ ]:
# ==================================================
# 使用PyTorch的nn.LSTM实现（推荐方式）
# ==================================================
num_inputs = vocab_size
num_epochs, lr = 500, 1  # 使用PyTorch实现时，学习率通常可以设置为1

# 创建PyTorch的LSTM层
# nn.LSTM内部已经实现了所有LSTM的计算逻辑
# 包含高度优化的CUDA实现，在GPU上运行非常高效
lstm_layer = nn.LSTM(num_inputs, num_hiddens)

# 使用封装好的RNN模型类（来自Rnn2.ipynb）
# 该类已经处理好LSTM的双状态(H, C)问题
model = RNN.RNNModel(lstm_layer, len(vocab))
model = model.to(device)

# 开始训练
# PyTorch的LSTM实现通常比从零实现更快、更稳定
# LSTM在处理长序列和长期依赖时表现优异
RNN.train_ch8(model, train_iter, vocab, lr, num_epochs, device)